# ARTI501 – Natural Language Processing
# Lab 2 – Text Pre-processing and Regular Expressions

**Task:** Apply text pre-processing and regular expressions to extract and analyze Twitter data regarding Apple company mentions — specifically, find the **total number of hashtags** and the **top 10 most frequently used hashtags**.

ALI ZUHAIR ALSAFFAR 2240005706


## 1. Objective / Learning Outcome

**CLO1:** Fundamentals of text pre-processing and regular expressions.

By the end of this notebook we will be able to:
- Load and explore a real-world Twitter dataset.
- Apply basic text pre-processing while keeping the information we actually need (hashtags).
- Write a regular expression that matches hashtags and use it with Python's `re` module.
- Count the total number of hashtags in the dataset and identify the 10 most frequently used ones.

## 2. Import Libraries

We only need a small set of libraries for this lab:
- **pandas** → load the CSV file and explore/manipulate it as a DataFrame.
- **re** → Python's built-in regular expression module, used to find hashtag patterns in the tweets.
- **collections.Counter** → count how many times each hashtag occurs.
- **matplotlib.pyplot** → a simple bar chart to visualize the top 10 hashtags at the end.

In [1]:
import pandas as pd
import re
from collections import Counter
import matplotlib.pyplot as plt

## 3. Load Dataset

**Dataset:** *Apple Twitter Sentiment texts*, as specified in the lab handout.

- **Kaggle page:** https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts

**How to get the file:**
1. Open the Kaggle link above (a free Kaggle account is required).
2. Click **Download** to get the dataset as a `.csv` file.
3. Place the downloaded CSV file in the **same folder as this notebook**.
4. Make sure the `DATASET_PATH` variable below matches the exact filename you downloaded (Kaggle usually names it `apple-twitter-sentiment-texts.csv`; rename it if needed, or just update the variable).

The code below loads the file with `pandas.read_csv()`. If the file is not found, it prints clear instructions instead of crashing, so the notebook stays easy to re-run once the dataset is in place.

In [2]:
# Path to the dataset file — update this if your downloaded file has a different name
DATASET_PATH = "apple-twitter-sentiment-texts.csv"

try:
    # Load the CSV file into a pandas DataFrame
    # encoding='latin-1' is used because Twitter datasets often contain special characters
    # (emojis, accented letters, etc.) that are not valid UTF-8
    df = pd.read_csv(DATASET_PATH, encoding="latin-1")
    print(f"Dataset loaded successfully from '{DATASET_PATH}'")
except FileNotFoundError:
    df = None
    print(f"Could not find '{DATASET_PATH}'.\n"
          "Please download the dataset from:\n"
          "https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts\n"
          "and place the CSV file in the same folder as this notebook "
          "(update DATASET_PATH above if the filename is different).")

Could not find 'apple-twitter-sentiment-texts.csv'.
Please download the dataset from:
https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts
and place the CSV file in the same folder as this notebook (update DATASET_PATH above if the filename is different).


## 4. Explore the Dataset

Before doing any pre-processing, we take a quick look at the dataset to understand its structure: how many rows/columns it has, what the columns are called, and — most importantly — which column holds the tweet text.

In [3]:
if df is not None:
    # Dataset shape: (number of rows, number of columns)
    print("Dataset shape:", df.shape)

    # Column names
    print("Columns:", list(df.columns))

    # A few sample rows
    display(df.head())

### Identifying the text column

Kaggle versions of this dataset have sometimes shipped with slightly different column names (e.g. `text`, `Text`, `tweet`, `Tweet`). The cell below automatically detects which column holds the tweet text, so the rest of the notebook keeps working even if the exact column name differs from what is expected.

In [4]:
if df is not None:
    # Candidate column names that commonly hold the tweet text in this dataset
    candidate_columns = ["text", "Text", "tweet", "Tweet", "tweet_text"]

    text_column = None
    for col in candidate_columns:
        if col in df.columns:
            text_column = col
            break

    if text_column is None:
        raise ValueError(f"Could not find a tweet-text column automatically. "
                          f"Available columns are: {list(df.columns)}. "
                          f"Please set 'text_column' manually.")

    print(f"Tweet text column detected: '{text_column}'")

## 5. Text Pre-processing

Before extracting hashtags we apply two simple pre-processing steps:

1. **Handle missing values** — some rows may have an empty/`NaN` tweet text; we convert everything to a string so `.lower()` never breaks on a missing value.
2. **Lowercase the text** — so that `#Apple`, `#APPLE`, and `#apple` are all treated as the *same* hashtag instead of three different ones.

**Important:** we deliberately do **not** remove punctuation or special characters at this stage, because doing so would also delete the `#` symbol — and we need it intact for the regular expression step that follows.

In [5]:
if df is not None:
    # Handle missing values first (fillna("") turns any NaN/missing tweet into an empty string),
    # then lowercase the text so hashtags that differ only by case are counted as the same hashtag
    df["text_clean"] = df[text_column].fillna("").astype(str).str.lower()

    # Preview the effect of pre-processing on the first few tweets
    display(df[[text_column, "text_clean"]].head())

## 6. Regular Expressions — Extracting Hashtags

A hashtag always starts with a literal `#`, followed by one or more letters, digits, or underscores.

**Pattern used:** `r'#\w+'`

- `#` → matches the literal hash symbol that starts every hashtag.
- `\w` → matches any "word character": `[a-zA-Z0-9_]`. This lets us capture hashtags that mix letters and numbers, e.g. `#iphone15`.
- `+` → "one or more": the hashtag must contain at least one character after the `#`, and the match keeps extending until it hits something that is *not* a letter/digit/underscore (a space, punctuation, emoji, etc.).

We apply `re.findall()` to every (already lowercased) tweet — `findall` returns *all* non-overlapping matches as a list, which is what we need since a single tweet can contain more than one hashtag.

In [6]:
if df is not None:
    # Regular expression: '#' followed by one-or-more word characters
    hashtag_pattern = r"#\w+"

    # For every tweet, extract the list of hashtags it contains (empty list if none)
    df["hashtags"] = df["text_clean"].apply(lambda tweet: re.findall(hashtag_pattern, tweet))

    # Preview a few tweets next to the hashtags extracted from them
    display(df[[text_column, "hashtags"]].head(10))

## 7. Total Number of Hashtags

Each row's `hashtags` column holds a *list* of hashtags found in that tweet. We flatten all of these per-tweet lists into one single list containing every hashtag **occurrence** in the whole dataset.

It's important to distinguish between:
- **Total hashtag occurrences** — every time a hashtag appears, counted separately (this is what the lab asks for).
- **Number of unique hashtags** — how many *distinct* hashtag words appear, regardless of repeats.

We report both, but the lab specifically asks for the **total** count.

In [7]:
if df is not None:
    # Flatten the list-of-lists into one flat list containing every hashtag occurrence
    all_hashtags = [tag for hashtag_list in df["hashtags"] for tag in hashtag_list]

    # Total number of hashtags = total occurrences (length of the flat list)
    total_hashtags = len(all_hashtags)

    # Number of unique hashtags, for comparison
    unique_hashtags = len(set(all_hashtags))

    print(f"Total number of hashtags (occurrences): {total_hashtags}")
    print(f"Number of unique hashtags: {unique_hashtags}")

## 8. Top 10 Most Frequent Hashtags

`collections.Counter` builds a frequency count of every unique hashtag in `all_hashtags`. Its `.most_common(n)` method returns the `n` most frequent hashtags, already sorted from most to least common.

In [8]:
if df is not None:
    # Count how many times each unique hashtag appears
    hashtag_counts = Counter(all_hashtags)

    # Get the 10 most frequent hashtags as (hashtag, count) pairs
    top_10 = hashtag_counts.most_common(10)

    # Display the results as a clean table
    top_10_df = pd.DataFrame(top_10, columns=["Hashtag", "Frequency"])
    top_10_df.index = range(1, len(top_10_df) + 1)
    top_10_df.index.name = "Rank"

    display(top_10_df)

### Visualizing the Top 10 Hashtags

A simple bar chart makes it easy to compare the relative frequency of the top hashtags at a glance.

In [9]:
if df is not None:
    plt.figure(figsize=(10, 5))
    plt.bar(top_10_df["Hashtag"], top_10_df["Frequency"], color="steelblue")
    plt.title("Top 10 Most Frequent Hashtags — Apple Twitter Sentiment Dataset")
    plt.xlabel("Hashtag")
    plt.ylabel("Frequency")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Results / Conclusion

**How hashtags were extracted:** Every tweet was lowercased (so `#Apple`/`#apple` count as the same hashtag) while keeping the `#` symbol intact, then the regular expression `r'#\w+'` was applied with `re.findall()` to pull out every hashtag in every tweet.

**Total number of hashtags found:** see the `total_hashtags` value printed in Section 7 (calculated directly from the dataset — not a fixed number, since it depends on the exact CSV file downloaded from Kaggle).

**Top 10 most frequent hashtags:** see the ranked table and bar chart in Section 8, generated from the actual dataset.

**What this demonstrates:** even a couple of simple pre-processing steps (handling missing values and lowercasing) combined with a short, well-designed regular expression are enough to reliably pull structured information (hashtags) out of noisy, unstructured tweet text. The resulting frequency counts give a quick view of which topics/tags (e.g. `#apple`, `#aapl`, `#iphone`) dominate the Apple-related conversation on Twitter in this dataset.